# US Revenue Forecast 결과 조회

ticker, forecast_date, indicator를 입력받아 해당 데이터를 조회 및 출력하는 노트북입니다.

## 1. 라이브러리 임포트 및 설정

In [91]:
import sys
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text
import warnings
from DATA.stock_invest_function import get_db_host
from typing import Optional

warnings.filterwarnings('ignore')

print("✓ 라이브러리 임포트 완료")

✓ 라이브러리 임포트 완료


## 2. DB 연결 설정

아래 셀에서 DB 연결 정보를 수정하세요.

In [2]:
# DB 연결 정보 설정
db_config = {
    'host': get_db_host(),  # 실행 시 get_db_host()로 설정
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

print("✓ DB 설정 완료")
print(f"  Host: {db_config['host']}:{db_config['port']}")
print(f"  Database: {db_config['database']}")

✓ DB 설정 완료
  Host: 192.168.0.230:3307
  Database: investar


## 3. RevenueForecastViewer 클래스 정의

In [3]:
class RevenueForecastViewer:
    """Revenue Forecast 데이터 조회 클래스"""

    def __init__(self, db_config):
        """
        Args:
            db_config: DB 연결 정보 딕셔너리
                - host, port, database, user, password
        """
        self.db_config = db_config
        self.engine = None
        self._connect_db()

    def _connect_db(self):
        """DB 연결"""
        try:
            conn_str = (
                f"mysql+pymysql://{self.db_config['user']}:{self.db_config['password']}@"
                f"{self.db_config['host']}:{self.db_config['port']}/"
                f"{self.db_config['database']}?charset=utf8mb4"
            )
            self.engine = create_engine(conn_str)
            print(f"✓ DB 연결 성공: {self.db_config['host']}:{self.db_config['port']}/{self.db_config['database']}")
        except Exception as e:
            print(f"✗ DB 연결 실패: {e}")
            raise

    def get_available_tickers(self):
        """사용 가능한 ticker 목록 조회"""
        query = """
        SELECT DISTINCT ticker
        FROM us_revenue_forecast_result
        ORDER BY ticker
        """
        try:
            with self.engine.connect() as conn:
                df = pd.read_sql(query, conn)
            return df['ticker'].tolist()
        except Exception as e:
            print(f"✗ Ticker 목록 조회 실패: {e}")
            return []

    def get_available_forecast_dates(self, ticker=None):
        """사용 가능한 forecast_date 목록 조회"""
        if ticker:
            query = """
            SELECT DISTINCT forecast_date
            FROM us_revenue_forecast_result
            WHERE ticker = :ticker
            ORDER BY forecast_date DESC
            """
            params = {'ticker': ticker}
        else:
            query = """
            SELECT DISTINCT forecast_date
            FROM us_revenue_forecast_result
            ORDER BY forecast_date DESC
            """
            params = {}

        try:
            with self.engine.connect() as conn:
                if params:
                    df = pd.read_sql(text(query), conn, params=params)
                else:
                    df = pd.read_sql(query, conn)
            return df['forecast_date'].tolist()
        except Exception as e:
            print(f"✗ Forecast date 목록 조회 실패: {e}")
            return []

    def get_available_indicators(self, ticker=None, forecast_date=None):
        """사용 가능한 indicator 목록 조회"""
        conditions = []
        params = {}

        if ticker:
            conditions.append("ticker = :ticker")
            params['ticker'] = ticker
        if forecast_date:
            conditions.append("forecast_date = :forecast_date")
            params['forecast_date'] = forecast_date

        where_clause = " WHERE " + " AND ".join(conditions) if conditions else ""

        query = f"""
        SELECT DISTINCT indicator
        FROM us_revenue_forecast_result
        {where_clause}
        ORDER BY indicator
        """

        try:
            with self.engine.connect() as conn:
                if params:
                    df = pd.read_sql(text(query), conn, params=params)
                else:
                    df = pd.read_sql(query, conn)
            return df['indicator'].tolist()
        except Exception as e:
            print(f"✗ Indicator 목록 조회 실패: {e}")
            return []

    def query_revenue_forecast(self, ticker, forecast_date, indicator=None):
        """
        Revenue forecast 데이터 조회

        Args:
            ticker: 종목 코드
            forecast_date: 예측 날짜 (YYYY-MM-DD)
            indicator: 지표명 (None이면 모든 지표)

        Returns:
            DataFrame
        """
        conditions = ["ticker = :ticker", "forecast_date = :forecast_date"]
        params = {'ticker': ticker, 'forecast_date': forecast_date}

        if indicator:
            conditions.append("indicator = :indicator")
            params['indicator'] = indicator

        where_clause = " AND ".join(conditions)

        query = f"""
        SELECT
            date,
            ticker,
            indicator,
            value,
            forecast_date,
            created_at,
            updated_at
        FROM us_revenue_forecast_result
        WHERE {where_clause}
        ORDER BY date, indicator
        """

        try:
            with self.engine.connect() as conn:
                df = pd.read_sql(text(query), conn, params=params)

            if df.empty:
                print(f"⚠ 조회된 데이터가 없습니다.")
                return df

            # 날짜 형식 변환
            df['date'] = pd.to_datetime(df['date'])
            df['forecast_date'] = pd.to_datetime(df['forecast_date'])

            return df

        except Exception as e:
            print(f"✗ 데이터 조회 실패: {e}")
            return pd.DataFrame()

    def display_results(self, df, output_format='wide'):
        """
        조회 결과 출력

        Args:
            df: 조회된 DataFrame
            output_format: 'wide' (pivot 테이블) 또는 'long' (원본 형태)
        """
        if df.empty:
            return

        print("\n" + "=" * 100)
        print(f"조회 결과: {len(df)} rows")
        print("=" * 100)

        if output_format == 'wide':
            # Wide format (indicator별로 컬럼 분리)
            pivot_df = df.pivot_table(
                index='date',
                columns='indicator',
                values='value',
                aggfunc='first'
            )
            pivot_df.index = pivot_df.index.strftime('%Y-%m-%d')

            print("\n[Wide Format - Pivot Table]")
            display(pivot_df)

            # 통계 정보
            print("\n[통계 정보]")
            display(pivot_df.describe())

        else:
            # Long format (원본)
            print("\n[Long Format - Original Data]")
            display_df = df.copy()
            display_df['date'] = display_df['date'].dt.strftime('%Y-%m-%d')
            display_df['forecast_date'] = display_df['forecast_date'].dt.strftime('%Y-%m-%d')
            display(display_df)

        # 메타 정보
        print("\n" + "-" * 100)
        print(f"Ticker: {df['ticker'].iloc[0]}")
        print(f"Forecast Date: {df['forecast_date'].iloc[0].strftime('%Y-%m-%d')}")
        print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
        print(f"Indicators: {', '.join(df['indicator'].unique())}")
        print("=" * 100)

    def save_to_csv(self, df, filename=None):
        """결과를 CSV 파일로 저장"""
        if df.empty:
            return

        if filename is None:
            ticker = df['ticker'].iloc[0]
            forecast_date = df['forecast_date'].iloc[0].strftime('%Y%m%d')
            filename = f"revenue_forecast_{ticker}_{forecast_date}.csv"

        try:
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"\n✓ CSV 파일 저장 완료: {filename}")
        except Exception as e:
            print(f"\n✗ CSV 파일 저장 실패: {e}")

print("✓ RevenueForecastViewer 클래스 정의 완료")

✓ RevenueForecastViewer 클래스 정의 완료


## 4. Viewer 초기화

In [4]:
# Viewer 객체 생성
viewer = RevenueForecastViewer(db_config)

✓ DB 연결 성공: 192.168.0.230:3307/investar


## 5. 사용 가능한 데이터 목록 조회

### 5.1 Ticker 목록

In [5]:
# 사용 가능한 ticker 목록 조회
tickers = viewer.get_available_tickers()
print(f"\n사용 가능한 Ticker: {len(tickers)}개")
print("-" * 50)
print(tickers[:20])  # 처음 20개만 출력


사용 가능한 Ticker: 844개
--------------------------------------------------
['A', 'AAP', 'AAPL', 'ABBV', 'ABG', 'ABM', 'ABNB', 'ABT', 'ACA', 'ACAD', 'ACHC', 'ACIW', 'ACLS', 'ACMR', 'ACN', 'ACT', 'ADBE', 'ADEA', 'ADI', 'ADM']


### 5.2 Forecast Date 목록

In [6]:
# 사용 가능한 forecast_date 목록 조회
dates = viewer.get_available_forecast_dates()
print(f"\n사용 가능한 Forecast Date: {len(dates)}개")
print("-" * 50)
for date in dates:
    print(date)


사용 가능한 Forecast Date: 3개
--------------------------------------------------
2025-11-06
2025-11-04
2025-11-02


### 5.3 Indicator 목록

In [7]:
# 사용 가능한 indicator 목록 조회
indicators = viewer.get_available_indicators()
print(f"\n사용 가능한 Indicator: {len(indicators)}개")
print("-" * 50)
for i, indicator in enumerate(indicators, 1):
    print(f"{i:2d}. {indicator}")


사용 가능한 Indicator: 4개
--------------------------------------------------
 1. revenue_billions_esq_forecast
 2. revenue_billions_lstm_forecast
 3. revenue_billions_prophet_forecast
 4. revenue_billions_sarima_noexog


## 6. 데이터 조회

### 6.1 기본 조회 (모든 지표)

In [19]:
# 조회할 ticker와 forecast_date 설정
ticker = 'MU'              # 여기를 원하는 ticker로 수정
forecast_date = '2025-11-06'  # 여기를 원하는 날짜로 수정

# 데이터 조회
df = viewer.query_revenue_forecast(ticker, forecast_date)

# 결과 출력 (Wide format)
if not df.empty:
    viewer.display_results(df, output_format='wide')


조회 결과: 744 rows

[Wide Format - Pivot Table]


indicator,revenue_billions_esq_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_sarima_noexog
date,,,,
1985-08-31,0.010000,0.010000,0.010000,0.010000
1985-11-30,0.010000,0.010000,0.010000,0.010000
1986-02-28,0.010000,0.010000,0.010000,0.010000
1986-05-31,0.010000,0.010000,0.010000,0.010000
1986-08-31,0.020000,0.020000,0.020000,0.020000
...,...,...,...,...
2026-10-31,16.748833,31.101959,8.185691,11.604942
2026-11-30,18.108541,31.101959,7.730339,11.662899
2026-12-31,18.108541,30.128878,7.730339,11.662899



[통계 정보]


indicator,revenue_billions_esq_forecast,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_sarima_noexog
count,186.000000,186.000000,186.000000,186.000000
mean,3.568230,4.654742,2.959192,3.276914
std,4.547077,8.041055,2.985577,3.616841
min,0.010000,0.010000,0.010000,0.010000
25%,0.730000,0.730000,0.730000,0.730000
50%,1.449000,1.499000,1.499000,1.499000
75%,4.775000,4.797500,4.797500,4.797500
max,19.468250,41.077103,11.310000,11.704056



----------------------------------------------------------------------------------------------------
Ticker: MU
Forecast Date: 2025-11-04
Date Range: 1985-08-31 ~ 2027-02-28
Indicators: revenue_billions_esq_forecast, revenue_billions_lstm_forecast, revenue_billions_prophet_forecast, revenue_billions_sarima_noexog


### 6.2 특정 지표만 조회

In [114]:
# 조회할 ticker, forecast_date, indicator 설정
ticker = 'AVGO'
forecast_date = '2025-11-06'
indicator = 'revenue_billions_prophet_forecast'  # 원하는 지표로 수정

 # 1. revenue_billions_esq_forecast
 # 2. revenue_billions_lstm_forecast
 # 3. revenue_billions_prophet_forecast
 # 4. revenue_billions_sarima_noexog

# 데이터 조회
df = viewer.query_revenue_forecast(ticker, forecast_date, indicator)

# 결과 출력 (Long format)
if not df.empty:
    viewer.display_results(df, output_format='long')


조회 결과: 63 rows

[Long Format - Original Data]


,date,ticker,indicator,value,forecast_date,created_at,updated_at
0,2015-01-31,AVGO,revenue_billions_prophet_forecast,1.640000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
1,2015-04-30,AVGO,revenue_billions_prophet_forecast,1.610000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
2,2015-07-31,AVGO,revenue_billions_prophet_forecast,1.740000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
3,2015-10-31,AVGO,revenue_billions_prophet_forecast,1.840000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
4,2016-01-31,AVGO,revenue_billions_prophet_forecast,1.770000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
...,...,...,...,...,...,...,...
58,2025-08-31,AVGO,revenue_billions_prophet_forecast,15.950000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
59,2025-11-30,AVGO,revenue_billions_prophet_forecast,-22.706378,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
60,2026-02-28,AVGO,revenue_billions_prophet_forecast,17.972521,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
61,2026-05-31,AVGO,revenue_billions_prophet_forecast,17.723125,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43



----------------------------------------------------------------------------------------------------
Ticker: AVGO
Forecast Date: 2025-11-06
Date Range: 2015-01-31 ~ 2026-08-31
Indicators: revenue_billions_prophet_forecast


In [115]:
df.tail(10)

,date,ticker,indicator,value,forecast_date,created_at,updated_at
53,2024-10-31,AVGO,revenue_billions_prophet_forecast,14.050000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
54,2025-01-31,AVGO,revenue_billions_prophet_forecast,14.916000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
55,2025-02-28,AVGO,revenue_billions_prophet_forecast,14.920000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
56,2025-04-30,AVGO,revenue_billions_prophet_forecast,15.004000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
57,2025-05-31,AVGO,revenue_billions_prophet_forecast,15.000000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
58,2025-08-31,AVGO,revenue_billions_prophet_forecast,15.950000,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
59,2025-11-30,AVGO,revenue_billions_prophet_forecast,-22.706378,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
60,2026-02-28,AVGO,revenue_billions_prophet_forecast,17.972521,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
61,2026-05-31,AVGO,revenue_billions_prophet_forecast,17.723125,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43
62,2026-08-31,AVGO,revenue_billions_prophet_forecast,18.369377,2025-11-06,2025-11-06 12:47:43,2025-11-06 12:47:43


In [116]:
def get_unique_values_from_db(db_info):
    """
    investar.us_psr_valuation_result 테이블에서
    forecast_date.unique(), indicator.unique() 값을 리스트로 반환
    """
    # ✅ DB 연결
    conn = pymysql.connect(
        host=db_info['host'],
        port=db_info['port'],
        user=db_info['user'],
        password=db_info['password'],
        database=db_info['database'],
        charset='utf8mb4'
    )

    try:
        # ✅ forecast_date unique 값 추출
        query_forecast = """
        SELECT DISTINCT forecast_date
        FROM us_psr_valuation_result
        ORDER BY forecast_date;
        """
        df_forecast = pd.read_sql(query_forecast, conn)
        forecast_dates = df_forecast['forecast_date'].astype(str).tolist()

        # ✅ indicator unique 값 추출
        query_indicator = """
        SELECT DISTINCT indicator
        FROM us_psr_valuation_result
        ORDER BY indicator;
        """
        df_indicator = pd.read_sql(query_indicator, conn)
        indicators = df_indicator['indicator'].tolist()

        return forecast_dates, indicators

    finally:
        conn.close()

def get_indicator_pivot(db_info: dict,
                        indicator: str = "sarima_valuation",
                        forecast_date: Optional[str] = None) -> pd.DataFrame:
    """
    us_psr_valuation_result에서 특정 indicator의 값을 불러와
    index=date, columns=ticker, values=value 피벗 테이블을 반환.

    Args:
        db_info: {'host','port','user','password','database'}
        indicator: 예) 'sarima_valuation'
        forecast_date: 'YYYY-MM-DD' 문자열. None이면 최신 forecast_date 자동 선택.

    Returns:
        pd.DataFrame: 날짜 × 티커 피벗 (값=indicator의 value)
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=True,
    )

    try:
        # 1️⃣ 최신 forecast_date 자동 선택
        if forecast_date is None:
            q_latest = """
                SELECT MAX(forecast_date) AS latest_fd
                FROM us_psr_valuation_result
                WHERE indicator = %s
            """
            latest_fd = pd.read_sql(q_latest, conn, params=[indicator])["latest_fd"][0]
            if pd.isna(latest_fd):
                raise ValueError(f"No rows found for indicator='{indicator}'.")
            forecast_date = str(latest_fd)

        # 2️⃣ 해당 forecast_date 자료 조회
        q = """
            SELECT `date`, `ticker`, `value`, `updated_ts`
            FROM us_psr_valuation_result
            WHERE indicator = %s AND forecast_date = %s
            ORDER BY `updated_ts` ASC
        """
        df = pd.read_sql(q, conn, params=[indicator, forecast_date])

        if df.empty:
            raise ValueError(f"No rows for indicator='{indicator}' on forecast_date='{forecast_date}'.")

        # 3️⃣ 중복 제거 (최신 updated_ts만)
        df = df.sort_values("updated_ts").drop_duplicates(subset=["date", "ticker"], keep="last")

        # 4️⃣ Pivot
        df["date"] = pd.to_datetime(df["date"])
        pivot = df.pivot(index="date", columns="ticker", values="value").sort_index()
        pivot = pivot.sort_index(axis=1)

        pivot.index.name = "date"
        pivot.columns.name = None

        print(f"✅ Loaded indicator='{indicator}', forecast_date={forecast_date}, shape={pivot.shape}")
        return pivot

    finally:
        conn.close()


def rank_changes_between(df_pivot: pd.DataFrame,
                         start_date: str = "2025-10-30",
                         end_date: str = "2026-12-31",
                         fill: str = "ffill") -> pd.DataFrame:
    """
    기간 [start_date, end_date] 내에서 각 컬럼의 변화율을 계산해 내림차순 정렬.

    변화율 정의: (마지막값 / 처음값 - 1) * 100 (%)

    Args:
        df_pivot : index=DatetimeIndex, columns=tickers
        start_date, end_date : 'YYYY-MM-DD' 문자열
        fill : 결측 처리 방식
               - "none": 원본 그대로(결측 있으면 해당 컬럼 NaN 결과)
               - "ffill": 기간 내 결측을 앞/뒤로 보간(ffill 후 bfill)

    Returns:
        pd.DataFrame(columns=["start_value","end_value","abs_change","pct_change_%"])
        인덱스=티커, pct_change_% 기준 내림차순 정렬
    """
    df = df_pivot.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # 1) 구간 슬라이스 (시작/끝 날짜가 정확히 없더라도 범위에 맞는 행 선택됨)
    period = df.loc[start_date:end_date]
    if period.empty:
        raise ValueError(f"No rows between {start_date} and {end_date}.")

    # 2) 결측 처리
    if fill == "ffill":
        period = period.ffill().bfill()
    elif fill == "none":
        pass
    else:
        raise ValueError("fill must be one of {'none','ffill'}")

    # 3) 각 컬럼의 처음/마지막 유효값 추출
    def first_valid(s: pd.Series):
        v = s.dropna()
        return v.iloc[0] if not v.empty else np.nan

    def last_valid(s: pd.Series):
        v = s.dropna()
        return v.iloc[-1] if not v.empty else np.nan

    first_vals = period.apply(first_valid, axis=0)
    last_vals  = period.apply(last_valid, axis=0)

    # 4) 변화율/절대변화 계산
    out = pd.DataFrame({
        "start_value": first_vals,
        "end_value": last_vals,
        "abs_change": last_vals - first_vals,
        "pct_change_%": (last_vals / first_vals - 1.0) * 100.0
    })

    # 시작/끝값이 없는 컬럼 제거
    out = out.dropna(subset=["start_value", "end_value"])

    # 5) 변화율 내림차순 정렬
    out = out.sort_values("pct_change_%", ascending=False)

    return out


# ── 사용 예시 ───────────────────────────────
# db_info = {
#     'host': 'localhost',
#     'port': 3307,
#     'user': 'stox7412',
#     'password': 'Apt106503!~',
#     'database': 'investar'
# }
# df_pivot = get_indicator_pivot(db_info, indicator='sarima_valuation')
# display(df_pivot.head())


# ── 사용 예시 ─────────────────────────────────────────
# 결과 표 (상승률 높은 순)
# rank_df = rank_changes_between(df_pivot,
#                                start_date="2025-10-30",
#                                end_date="2026-12-31",
#                                fill="ffill")   # 결측이 많다면 'ffill' 권장
# print(rank_df.head(20))        # 상위 20개 확인

In [117]:
import pymysql

forecast_dates, indicators = get_unique_values_from_db(db_config)
print("📅 forecast_date.unique():", forecast_dates)
print("📊 indicator.unique():", indicators)

📅 forecast_date.unique(): ['2025-11-06']
📊 indicator.unique(): ['es_valuation', 'lstm_valuation', 'prophet_valuation', 'PSR_es_forecast', 'PSR_prophet_forecast_noexog', 'PSR_ttm_lstm_forecast', 'PSR_ttm_sarima_forecast', 'revenue_billions_avg_of_4_ttm', 'revenue_billions_esq_forecast_ttm', 'revenue_billions_lstm_forecast_ttm', 'revenue_billions_prophet_forecast_ttm', 'revenue_billions_sarima_noexog_ttm', 'sarima_valuation']


In [118]:
df_pivot = get_indicator_pivot(db_config, indicator='sarima_valuation')

✅ Loaded indicator='sarima_valuation', forecast_date=2025-11-06, shape=(39, 813)


In [119]:
psr_pivot = get_indicator_pivot(db_config, indicator='PSR_ttm_sarima_forecast')
psr_pivot[ticker].tail(36)

✅ Loaded indicator='PSR_ttm_sarima_forecast', forecast_date=2025-11-06, shape=(39, 813)


date
2024-05-31          NaN
2024-06-30          NaN
2024-07-31          NaN
2024-08-31          NaN
2024-09-30          NaN
2024-10-31          NaN
2024-11-30          NaN
2024-12-31          NaN
2025-01-31          NaN
2025-02-28          NaN
2025-03-31          NaN
2025-04-30          NaN
2025-05-31    19.375914
2025-06-30    21.711513
2025-07-31    23.100896
2025-08-31    23.359756
2025-09-30    25.918272
2025-10-31    28.588023
2025-11-30    28.517901
2025-12-31    28.448003
2026-01-31    28.378327
2026-02-28    28.308872
2026-03-31    28.239638
2026-04-30    28.170624
2026-05-31    28.101829
2026-06-30    28.033252
2026-07-31    27.964892
2026-08-31    27.896749
2026-09-30    27.828822
2026-10-31    27.761110
2026-11-30    27.693612
2026-12-31    27.626327
2027-01-31    27.559255
2027-02-28    27.492395
2027-03-31    27.425745
2027-04-30    27.359306
Name: AVGO, dtype: float64

In [120]:
ttm_pivot = get_indicator_pivot(db_config, indicator='revenue_billions_sarima_noexog_ttm')
ttm_pivot[ticker].tail(36)

✅ Loaded indicator='revenue_billions_sarima_noexog_ttm', forecast_date=2025-11-06, shape=(39, 813)


date
2024-05-31          NaN
2024-06-30          NaN
2024-07-31          NaN
2024-08-31          NaN
2024-09-30          NaN
2024-10-31          NaN
2024-11-30          NaN
2024-12-31          NaN
2025-01-31          NaN
2025-02-28          NaN
2025-03-31          NaN
2025-04-30          NaN
2025-05-31    59.840000
2025-06-30    59.840000
2025-07-31    60.539485
2025-08-31    61.569485
2025-09-30    61.569485
2025-10-31    62.908900
2025-11-30    44.874991
2025-12-31    44.874991
2026-01-31    46.338041
2026-02-28    48.268485
2026-03-31    48.268485
2026-04-30    49.722198
2026-05-31    71.066938
2026-06-30    71.066938
2026-07-31    72.576009
2026-08-31    73.809420
2026-09-30    73.809420
2026-10-31    75.430490
2026-11-30    75.430490
2026-12-31    75.430490
2027-01-31          NaN
2027-02-28          NaN
2027-03-31          NaN
2027-04-30          NaN
Name: AVGO, dtype: float64

In [121]:
rank_df = rank_changes_between(df_pivot,
                               start_date="2025-10-30",
                               end_date="2026-12-31",
                               fill="ffill")   # 결측이 많다면 'ffill' 권장
print(rank_df.head(40))        # 상위 20개 확인

      start_value    end_value  abs_change  pct_change_%
MDU     -0.115467    -1.357774   -1.242307   1075.897867
MOH      9.757350    36.850544   27.093193    277.669577
INDB     4.805152    17.527172   12.722020    264.757940
SITC     0.116923     0.345435    0.228512    195.438009
BG      15.487802    41.765050   26.277249    169.664158
CLSK     2.536813     6.117791    3.580979    141.160558
SAFE     1.448291     3.361679    1.913388    132.113531
PLAY     0.495620     1.147161    0.651541    131.459706
CZR      4.105833     9.418468    5.312634    129.392353
UNIT     1.115966     2.546387    1.430421    128.177769
TGTX     5.005556    11.021633    6.016077    120.188005
VTOL     1.154295     2.303669    1.149374     99.573620
NAVI     1.261963     2.428004    1.166041     92.398938
SPNT     2.811745     5.331395    2.519650     89.611611
INTU   163.234484   309.461909  146.227425     89.581210
LGND     4.784000     8.973784    4.189784     87.579096
PENN     2.404173     4.255102 

In [127]:
rank_df.loc['GLW']

start_value     75.998343
end_value       79.931956
abs_change       3.933613
pct_change_%     5.175919
Name: GLW, dtype: float64